# SHIPP Stage 13 — Marketplace Analytics

**Owner:** Raneem  
**Priority:** P0  
**Current gate:** 13.0 — Analytics CDF source readiness

## Goal

Verify that the two operational Agent-write tables:

- `shipp.saved_items`
- `shipp.agent_activity`

are arriving in Bronze through Lakebase CDF before we build
`gold_marketplace_metrics`.

**This gate is read-only.** It does not create or modify analytics data.

### PASS condition

Both expected Bronze CDF tables must:

1. exist,
2. contain at least one row,
3. expose the SHIPP Lakebase CDF ordering metadata:
   - `_pg_change_type`
   - `_pg_lsn`
   - `_sort_by`


In [0]:
%uv pip install "psycopg[binary]>=3.2,<4"

In [0]:
%run ../common/shipp_silver_lib

In [0]:
# ============================================================
# STAGE 13 — VALIDATION ONLY
# Controlled Agent save
# DO NOT ADD THIS NOTEBOOK TO PIPELINE 90
# ============================================================

print("=== STAGE 13 — CONTROLLED AGENT SAVE ===")

import os
import sys
import psycopg

from databricks.sdk import WorkspaceClient


ENDPOINT_NAME = (
    "projects/shipp-capstone-v1/"
    "branches/production/endpoints/primary"
)

PGHOST = (
    "ep-dry-flower-d17p00tu."
    "database.us-west-2.cloud.databricks.com"
)

PGDATABASE = "databricks_postgres"
LAKEBASE_SCHEMA = "shipp"

w = WorkspaceClient()
PGUSER = w.current_user.me().user_name


def connect():

    credential = (
        w.postgres.generate_database_credential(
            endpoint=ENDPOINT_NAME
        )
    )

    return psycopg.connect(
        host=PGHOST,
        port=5432,
        dbname=PGDATABASE,
        user=PGUSER,
        password=credential.token,
        sslmode="require",
        connect_timeout=10,
    )


# ------------------------------------------------------------
# Existing Agent implementation
# ------------------------------------------------------------

agent_src = os.path.join(
    REPO_ROOT,
    "agent",
    "agent_ship",
    "src",
)

if agent_src not in sys.path:
    sys.path.insert(0, agent_src)

from shipp.agent.lakebase import LakebaseRepo

lakebase = LakebaseRepo(
    connect,
    LAKEBASE_SCHEMA,
)


# ------------------------------------------------------------
# Analytics baseline
# ------------------------------------------------------------

baseline = (
    spark.table(GOLD_MARKETPLACE_METRICS)
    .filter(F.col("metric_key") == "marketplace_current")
    .first()
)

assert baseline is not None

X = int(
    baseline["saved_items_count"]
)

EXPECTED_AFTER = X + 1

print("Baseline X      =", X)
print("Expected after  =", EXPECTED_AFTER)


# ------------------------------------------------------------
# Current Lakebase count
# ------------------------------------------------------------

with connect() as conn, conn.cursor() as cur:

    cur.execute(
        f"""
        SELECT count(*)
        FROM {LAKEBASE_SCHEMA}.saved_items
        """
    )

    lakebase_before = int(
        cur.fetchone()[0]
    )


print(
    "Lakebase before =",
    lakebase_before,
)


# ------------------------------------------------------------
# Recovery protection
# ------------------------------------------------------------

if lakebase_before == EXPECTED_AFTER:

    print(
        "A controlled save already exists."
    )

    print(
        "NO additional save executed."
    )


elif lakebase_before == X:

    candidates = (
        spark.table(GOLD_CANDIDATE_MATCHES)
        .select(
            "request_id",
            "listing_id",
            "match_score",
        )
        .orderBy(
            "request_id",
            F.col("match_score").desc(),
            "listing_id",
        )
        .collect()
    )

    selected = None

    for c in candidates:

        request_id = str(
            c["request_id"]
        )

        listing_id = str(
            c["listing_id"]
        )

        request_state = lakebase.get_request(
            request_id
        )

        listing_state = lakebase.get_listing(
            listing_id
        )

        if request_state is None:
            continue

        if listing_state is None:
            continue

        if str(listing_state.status).upper() != "AVAILABLE":
            continue

        user_id = str(
            request_state.requester_id
        )

        with connect() as conn, conn.cursor() as cur:

            cur.execute(
                f"""
                SELECT count(*)
                FROM {LAKEBASE_SCHEMA}.saved_items
                WHERE user_id = %s
                  AND request_id = %s
                  AND listing_id = %s
                """,
                (
                    user_id,
                    request_id,
                    listing_id,
                ),
            )

            existing = int(
                cur.fetchone()[0]
            )

        if existing == 0:

            selected = (
                user_id,
                request_id,
                listing_id,
            )

            break


    assert selected is not None, (
        "BLOCKED: no unsaved eligible candidate."
    )

    USER_ID, REQUEST_ID, LISTING_ID = selected

    print()
    print("Selected:")
    print("user_id    =", USER_ID)
    print("request_id =", REQUEST_ID)
    print("listing_id =", LISTING_ID)


    result = lakebase.save_item(
        user_id=USER_ID,
        request_id=REQUEST_ID,
        listing_id=LISTING_ID,
    )

    print(
        "save_item result =",
        result,
    )


    status = getattr(
        result.status,
        "value",
        str(result.status),
    )

    assert status == "SUCCESS", (
        f"Expected SUCCESS, got {result}"
    )

else:

    raise AssertionError(
        f"Unexpected state: "
        f"Gold={X}, Lakebase={lakebase_before}"
    )


# ------------------------------------------------------------
# Final Lakebase confirmation
# ------------------------------------------------------------

with connect() as conn, conn.cursor() as cur:

    cur.execute(
        f"""
        SELECT count(*)
        FROM {LAKEBASE_SCHEMA}.saved_items
        """
    )

    lakebase_after = int(
        cur.fetchone()[0]
    )


print()
print("X              =", X)
print("Lakebase after =", lakebase_after)
print("Expected       =", EXPECTED_AFTER)

assert lakebase_after == EXPECTED_AFTER

print()
print(
    "PASS — controlled Agent save ready "
    "for downstream validation."
)

In [0]:
# ============================================================
# STAGE 13 VALIDATION — CONTROLLED SAVE
# MANUAL VALIDATION ONLY
# ============================================================

print("=== STAGE 13 — CONTROLLED SAVE VALIDATION ===")

import os
import sys
import psycopg

from databricks.sdk import WorkspaceClient


# ------------------------------------------------------------
# Lakebase
# ------------------------------------------------------------

ENDPOINT_NAME = (
    "projects/shipp-capstone-v1/"
    "branches/production/endpoints/primary"
)

PGHOST = (
    "ep-dry-flower-d17p00tu."
    "database.us-west-2.cloud.databricks.com"
)

PGDATABASE = "databricks_postgres"
LAKEBASE_SCHEMA = "shipp"

w = WorkspaceClient()
PGUSER = w.current_user.me().user_name


def connect():

    credential = (
        w.postgres.generate_database_credential(
            endpoint=ENDPOINT_NAME
        )
    )

    return psycopg.connect(
        host=PGHOST,
        port=5432,
        dbname=PGDATABASE,
        user=PGUSER,
        password=credential.token,
        sslmode="require",
        connect_timeout=10,
    )


# ------------------------------------------------------------
# Existing validated Agent
# ------------------------------------------------------------

agent_src = os.path.join(
    REPO_ROOT,
    "agent",
    "agent_ship",
    "src",
)

if agent_src not in sys.path:
    sys.path.insert(0, agent_src)


from shipp.agent.contracts import ActionStatus
from shipp.agent.lakebase import LakebaseRepo


lakebase = LakebaseRepo(
    connect,
    LAKEBASE_SCHEMA,
)


# ------------------------------------------------------------
# Baseline X from Gold
# ------------------------------------------------------------

baseline = (
    spark.table(GOLD_MARKETPLACE_METRICS)
    .filter(
        F.col("metric_key")
        == "marketplace_current"
    )
    .first()
)

assert baseline is not None

X = int(
    baseline["saved_items_count"]
)

print("Gold baseline X =", X)


# ------------------------------------------------------------
# Current Lakebase count
# ------------------------------------------------------------

with connect() as conn, conn.cursor() as cur:

    cur.execute(
        f"""
        SELECT count(*)
        FROM {LAKEBASE_SCHEMA}.saved_items
        """
    )

    lakebase_count = int(
        cur.fetchone()[0]
    )


print(
    "Current Lakebase count =",
    lakebase_count,
)


# ------------------------------------------------------------
# RECOVERY CASE
#
# If interrupted Cell 11 already created the save,
# DO NOT CREATE ANOTHER.
# ------------------------------------------------------------

if lakebase_count == X + 1:

    print()
    print(
        "RECOVERY: controlled save already exists."
    )

    print(
        "No additional save will be executed."
    )

    EXPECTED_AFTER = X + 1


# ------------------------------------------------------------
# NORMAL CASE
# ------------------------------------------------------------

elif lakebase_count == X:

    print()
    print(
        "No previous controlled save detected."
    )

    gold_candidates = (
        spark.table(
            GOLD_CANDIDATE_MATCHES
        )
        .select(
            "request_id",
            "listing_id",
            "match_score",
        )
        .orderBy(
            "request_id",
            F.col("match_score").desc(),
            "listing_id",
        )
        .collect()
    )

    selected = None

    for c in gold_candidates:

        request_id = str(
            c["request_id"]
        )

        listing_id = str(
            c["listing_id"]
        )

        request_state = (
            lakebase.get_request(
                request_id
            )
        )

        listing_state = (
            lakebase.get_listing(
                listing_id
            )
        )

        if (
            request_state is None
            or listing_state is None
        ):
            continue

        if str(
            listing_state.status
        ).upper() != "AVAILABLE":
            continue

        user_id = str(
            request_state.requester_id
        )

        with connect() as conn, conn.cursor() as cur:

            cur.execute(
                f"""
                SELECT count(*)
                FROM {LAKEBASE_SCHEMA}.saved_items
                WHERE user_id = %s
                  AND request_id = %s
                  AND listing_id = %s
                """,
                (
                    user_id,
                    request_id,
                    listing_id,
                ),
            )

            existing = int(
                cur.fetchone()[0]
            )

        if existing == 0:

            selected = (
                user_id,
                request_id,
                listing_id,
            )

            break


    assert selected is not None, (
        "BLOCKED: no unsaved candidate found."
    )


    USER_ID, REQUEST_ID, LISTING_ID = (
        selected
    )


    print()
    print("Controlled candidate:")
    print("user_id    =", USER_ID)
    print("request_id =", REQUEST_ID)
    print("listing_id =", LISTING_ID)


    result = lakebase.save_item(
        user_id=USER_ID,
        request_id=REQUEST_ID,
        listing_id=LISTING_ID,
    )


    print()
    print(
        "save_item result =",
        result,
    )


    status = getattr(
        result.status,
        "value",
        str(result.status),
    )


    assert status == "SUCCESS", (
        f"FAILED: expected SUCCESS, got {result}"
    )


    with connect() as conn, conn.cursor() as cur:

        cur.execute(
            f"""
            SELECT count(*)
            FROM {LAKEBASE_SCHEMA}.saved_items
            """
        )

        after_count = int(
            cur.fetchone()[0]
        )


    assert after_count == X + 1, (
        f"FAILED: expected {X + 1}, "
        f"found {after_count}"
    )


    EXPECTED_AFTER = X + 1


else:

    raise AssertionError(
        "Unexpected state: "
        f"Gold X={X}, Lakebase={lakebase_count}"
    )


print()
print("===================================")
print("STAGE 13 CONTROLLED SAVE RESULT")
print("===================================")

print("Before X       =", X)
print("Expected after =", EXPECTED_AFTER)

print()
print(
    "PASS — controlled save state ready "
    "for analytics validation."
)

In [0]:
# ============================================================
# STAGE 13 — VALIDATION ONLY
# Controlled Agent save
# DO NOT ADD THIS NOTEBOOK TO PIPELINE 90
# ============================================================

print("=== STAGE 13 — CONTROLLED AGENT SAVE ===")

import os
import sys
import psycopg

from databricks.sdk import WorkspaceClient


ENDPOINT_NAME = (
    "projects/shipp-capstone-v1/"
    "branches/production/endpoints/primary"
)

PGHOST = (
    "ep-dry-flower-d17p00tu."
    "database.us-west-2.cloud.databricks.com"
)

PGDATABASE = "databricks_postgres"
LAKEBASE_SCHEMA = "shipp"

w = WorkspaceClient()
PGUSER = w.current_user.me().user_name


def connect():

    credential = (
        w.postgres.generate_database_credential(
            endpoint=ENDPOINT_NAME
        )
    )

    return psycopg.connect(
        host=PGHOST,
        port=5432,
        dbname=PGDATABASE,
        user=PGUSER,
        password=credential.token,
        sslmode="require",
        connect_timeout=10,
    )


# ------------------------------------------------------------
# Existing Agent implementation
# ------------------------------------------------------------

agent_src = os.path.join(
    REPO_ROOT,
    "agent",
    "agent_ship",
    "src",
)

if agent_src not in sys.path:
    sys.path.insert(0, agent_src)

from shipp.agent.lakebase import LakebaseRepo

lakebase = LakebaseRepo(
    connect,
    LAKEBASE_SCHEMA,
)


# ------------------------------------------------------------
# Analytics baseline
# ------------------------------------------------------------

baseline = (
    spark.table(GOLD_MARKETPLACE_METRICS)
    .filter(F.col("metric_key") == "marketplace_current")
    .first()
)

assert baseline is not None

X = int(
    baseline["saved_items_count"]
)

EXPECTED_AFTER = X + 1

print("Baseline X      =", X)
print("Expected after  =", EXPECTED_AFTER)


# ------------------------------------------------------------
# Current Lakebase count
# ------------------------------------------------------------

with connect() as conn, conn.cursor() as cur:

    cur.execute(
        f"""
        SELECT count(*)
        FROM {LAKEBASE_SCHEMA}.saved_items
        """
    )

    lakebase_before = int(
        cur.fetchone()[0]
    )


print(
    "Lakebase before =",
    lakebase_before,
)


# ------------------------------------------------------------
# Recovery protection
# ------------------------------------------------------------

if lakebase_before == EXPECTED_AFTER:

    print(
        "A controlled save already exists."
    )

    print(
        "NO additional save executed."
    )


elif lakebase_before == X:

    candidates = (
        spark.table(GOLD_CANDIDATE_MATCHES)
        .select(
            "request_id",
            "listing_id",
            "match_score",
        )
        .orderBy(
            "request_id",
            F.col("match_score").desc(),
            "listing_id",
        )
        .collect()
    )

    selected = None

    for c in candidates:

        request_id = str(
            c["request_id"]
        )

        listing_id = str(
            c["listing_id"]
        )

        request_state = lakebase.get_request(
            request_id
        )

        listing_state = lakebase.get_listing(
            listing_id
        )

        if request_state is None:
            continue

        if listing_state is None:
            continue

        if str(listing_state.status).upper() != "AVAILABLE":
            continue

        user_id = str(
            request_state.requester_id
        )

        with connect() as conn, conn.cursor() as cur:

            cur.execute(
                f"""
                SELECT count(*)
                FROM {LAKEBASE_SCHEMA}.saved_items
                WHERE user_id = %s
                  AND request_id = %s
                  AND listing_id = %s
                """,
                (
                    user_id,
                    request_id,
                    listing_id,
                ),
            )

            existing = int(
                cur.fetchone()[0]
            )

        if existing == 0:

            selected = (
                user_id,
                request_id,
                listing_id,
            )

            break


    assert selected is not None, (
        "BLOCKED: no unsaved eligible candidate."
    )

    USER_ID, REQUEST_ID, LISTING_ID = selected

    print()
    print("Selected:")
    print("user_id    =", USER_ID)
    print("request_id =", REQUEST_ID)
    print("listing_id =", LISTING_ID)


    result = lakebase.save_item(
        user_id=USER_ID,
        request_id=REQUEST_ID,
        listing_id=LISTING_ID,
    )

    print(
        "save_item result =",
        result,
    )


    status = getattr(
        result.status,
        "value",
        str(result.status),
    )

    assert status == "SUCCESS", (
        f"Expected SUCCESS, got {result}"
    )

else:

    raise AssertionError(
        f"Unexpected state: "
        f"Gold={X}, Lakebase={lakebase_before}"
    )


# ------------------------------------------------------------
# Final Lakebase confirmation
# ------------------------------------------------------------

with connect() as conn, conn.cursor() as cur:

    cur.execute(
        f"""
        SELECT count(*)
        FROM {LAKEBASE_SCHEMA}.saved_items
        """
    )

    lakebase_after = int(
        cur.fetchone()[0]
    )


print()
print("X              =", X)
print("Lakebase after =", lakebase_after)
print("Expected       =", EXPECTED_AFTER)

assert lakebase_after == EXPECTED_AFTER

print()
print(
    "PASS — controlled Agent save ready "
    "for downstream validation."
)

In [0]:
# ============================================================
# STAGE 13 — FINAL ANALYTICS VALIDATION
# Proof: X -> X+1 -> X+1
# ============================================================

print(
    "=== STAGE 13 — FINAL X -> X+1 -> X+1 VALIDATION ==="
)

import time


# ------------------------------------------------------------
# Wait for Bronze
# ------------------------------------------------------------

deadline = time.time() + 180

bronze_count = None


while time.time() < deadline:

    try:
        spark.sql(
            f"REFRESH TABLE {BRONZE_SAVED_ITEMS_HISTORY}"
        )
    except Exception:
        pass


    bronze_df = spark.table(
        BRONZE_SAVED_ITEMS_HISTORY
    )


    (
        _latest,
        bronze_current,
        _deleted,
    ) = reconstruct_current_state(
        bronze_df=bronze_df,
        key="saved_item_id",
    )


    bronze_count = bronze_current.count()

    print(
        "Bronze current saved_items =",
        bronze_count,
    )


    if bronze_count == EXPECTED_AFTER:
        break

    time.sleep(5)


assert bronze_count == EXPECTED_AFTER, (
    "BLOCKED: save has not reached Bronze."
)


print(
    "PASS — new save reached Bronze."
)


# ------------------------------------------------------------
# Run production Notebook 60
# ------------------------------------------------------------

print()
print("Running Notebook 60 — first refresh...")


run1 = dbutils.notebook.run(
    "./60_gold_marketplace_metrics",
    1800,
)


after_first = (
    spark.table(GOLD_MARKETPLACE_METRICS)
    .filter(
        F.col("metric_key")
        == "marketplace_current"
    )
    .first()
)


after_first_count = int(
    after_first["saved_items_count"]
)


print(
    "After first analytics run =",
    after_first_count,
)


assert after_first_count == EXPECTED_AFTER


# Save logical state excluding processing timestamp

snapshot_1 = [
    r.asDict(recursive=True)
    for r in (
        spark.table(GOLD_MARKETPLACE_METRICS)
        .drop("analytics_processed_at")
        .orderBy("metric_key")
        .collect()
    )
]


# ------------------------------------------------------------
# Run Notebook 60 again — replay proof
# ------------------------------------------------------------

print()
print("Running Notebook 60 — replay...")


run2 = dbutils.notebook.run(
    "./60_gold_marketplace_metrics",
    1800,
)


final_df = spark.table(
    GOLD_MARKETPLACE_METRICS
)


final_row = (
    final_df
    .filter(
        F.col("metric_key")
        == "marketplace_current"
    )
    .first()
)


final_count = int(
    final_row["saved_items_count"]
)


duplicate_keys = (
    final_df
    .groupBy("metric_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)


snapshot_2 = [
    r.asDict(recursive=True)
    for r in (
        final_df
        .drop("analytics_processed_at")
        .orderBy("metric_key")
        .collect()
    )
]


assert final_count == EXPECTED_AFTER
assert duplicate_keys == 0
assert snapshot_1 == snapshot_2


print()
print("===================================")
print("FINAL STAGE 13 PROOF")
print("===================================")

print("Before       =", X)
print("After save   =", after_first_count)
print("After replay =", final_count)
print("Duplicates   =", duplicate_keys)

print()
print(
    f"FINAL PROOF: "
    f"{X} -> {after_first_count} -> {final_count}"
)

print()
print(
    "PASS STAGE 13 — Analytics validated: "
    "new Lakebase action appeared downstream "
    "and replay did not double-count."
)

display(final_df)